# Quiz AI Module — Phi-3.5-mini on Colab T4 GPU

This notebook replaces the previous Qwen2.5-3B setup. It uses **microsoft/Phi-3.5-mini-instruct** for better:
- MCQ distractor quality (no more ambiguous/overlapping wrong answers)
- Structured JSON output (fewer structural failures)
- Speed on T4 GPU (~3-8 seconds vs 25-40 seconds)
- Instruction following (proper difficulty levels)

**Steps:** Hardware Check - Install - Model Test - Benchmark - Full Pipeline - (Optional) LoRA Training - API Server for Flutter

---
**First time?** Upload `quiz-project-current.zip` to the Colab Files panel before running Cell 2.

## Cell 1: Hardware Check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected!\n'
        'Go to Runtime > Change runtime type > Select T4 GPU > Save\n'
        'Then re-run this cell.'
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')

if vram_gb < 12:
    print(f'Warning: {vram_gb:.1f} GB may be tight. T4 (16 GB) is recommended.')
else:
    print('Sufficient VRAM for Phi-3.5-mini in float16')

print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')

## Cell 2: Install Dependencies and Setup Project

In [ ]:
import os, subprocess, zipfile
from pathlib import Path

subprocess.check_call(['pip', 'install', '-q',
    'transformers==4.57.6', 'peft==0.18.1', 'accelerate>=1.10,<2',
    'pydantic>=2.6,<3', 'python-dotenv>=1,<2', 'pymupdf>=1.24,<2',
    'python-docx>=1.1,<2', 'fastapi>=0.115,<1', 'uvicorn>=0.29,<1',
    'python-multipart>=0.0.18', 'ipywidgets>=8.1,<9', 'httpx>=0.27,<1',
    'pyngrok'])

project_dir = Path('/content/quiz_ai_module')
zip_path = Path('/content/quiz-project-current.zip')

if zip_path.exists():
    if project_dir.exists():
        import shutil
        shutil.rmtree(project_dir)
    project_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(project_dir)
    print(f'Extracted project to {project_dir}')
elif project_dir.exists():
    print(f'Using existing project at {project_dir}')
else:
    raise FileNotFoundError(
        'Upload quiz-project-current.zip to the Files panel first!'
    )

os.chdir(project_dir)
import sys
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

os.environ['AI_PROVIDER'] = 'huggingface'
os.environ['HF_MODEL'] = 'microsoft/Phi-3.5-mini-instruct'

print('Dependencies installed and project ready')
print(f'Working directory: {os.getcwd()}')

## Cell 3: Load Phi-3.5-mini-instruct

In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'microsoft/Phi-3.5-mini-instruct'

print(f'Loading {MODEL_ID}...')
load_start = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='sdpa',
)
model.eval()

load_time = time.perf_counter() - load_start
vram_used = torch.cuda.memory_allocated() / 1e9

print(f'\nModel loaded in {load_time:.1f} seconds')
print(f'VRAM used: {vram_used:.1f} GB')
print(f'Device: {next(model.parameters()).device}')

## Cell 4: Quick Model Sanity Test

In [ ]:
import json, time

test_passage = """The Sun is the closest star to Earth. It provides light and heat.
The Moon orbits Earth approximately every 27.3 days. Earth orbits the Sun once every 365.25 days.
Mars is known as the Red Planet because of iron oxide on its surface."""

test_system = """You create accurate reading-comprehension MCQ quizzes from study material.
Return ONLY valid JSON with this exact format:
{"questions":[{"q":"question text?","a":"correct answer","d":["wrong1","wrong2","wrong3"],"source":"exact quote from passage"}]}
Rules:
- a must be a verbatim substring from the passage
- d must contain 3 CLEARLY wrong alternatives (no synonyms or overlapping terms)
- All 4 options must be from the same category
- No markdown, no commentary"""

test_user = f"""PASSAGE:
\"\"\"
{test_passage}
\"\"\"

Generate exactly 1 easy MCQ question. Return only the JSON."""

messages = [
    {'role': 'user', 'content': f'{test_system}\n\n{test_user}'}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

start = time.perf_counter()
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
elapsed = time.perf_counter() - start

generated = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
new_tokens = output.shape[1] - inputs.input_ids.shape[1]

print(f'Generated {new_tokens} tokens in {elapsed:.1f}s ({new_tokens/elapsed:.1f} tok/s)')
print(f'\nRaw output:\n{generated}')

try:
    text = generated.strip()
    if text.startswith('```'):
        text = text.split('\n', 1)[1].rsplit('```', 1)[0].strip()
    data = json.loads(text)
    print(f'\nValid JSON!')
    print(json.dumps(data, indent=2))
except json.JSONDecodeError as e:
    print(f'\nJSON parse error: {e}')
    print('The model output is not valid JSON. Check raw output above.')

## Cell 5: Run Full Pipeline Benchmark

Uses the actual quiz generation agent to generate 3 and 5 questions, measuring speed and quality.

In [ ]:
import json, os, sys, time
from pathlib import Path

project_dir = Path('/content/quiz_ai_module')
os.chdir(project_dir)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

os.environ['AI_PROVIDER'] = 'huggingface'
os.environ['HF_MODEL'] = 'microsoft/Phi-3.5-mini-instruct'

from ai_quiz_agent.providers import get_hf_provider
get_hf_provider.cache_clear()

from ai_quiz_agent.agents.quiz_generation_agent import QuizGenerationAgent
from ai_quiz_agent.generation_control import bounded_generation
from ai_quiz_agent.schemas.models import Difficulty, QuestionType

notes_path = project_dir / 'examples' / 'sample-study-notes.txt'
if not notes_path.exists():
    raise FileNotFoundError(f'Missing {notes_path}. Did quiz-project-current.zip extract correctly?')
text = notes_path.read_text(encoding='utf-8-sig')
print(f'Study notes: {len(text)} characters\n')

agent = QuizGenerationAgent(max_retries=1)
results = []

for count in [3, 5]:
    for difficulty in ['easy', 'medium']:
        print(f'--- {count} {difficulty} questions ---')
        start = time.perf_counter()
        try:
            quiz = bounded_generation(agent.generate_quiz)(
                text, Difficulty(difficulty), count, QuestionType.MCQ,
                'benchmark', 'benchmark'
            )
            elapsed = time.perf_counter() - start
            result = {
                'count': count, 'difficulty': difficulty,
                'status': 'passed', 'seconds': round(elapsed, 1),
                'questions': []
            }
            for q in quiz.questions:
                qdata = {
                    'question': q.question,
                    'correct': q.correct_answer,
                    'options': q.options,
                    'source': q.source_quote[:80] + '...' if len(q.source_quote) > 80 else q.source_quote
                }
                print(f'  Q: {q.question}')
                print(f'  A: {q.correct_answer}')
                print(f'  Options: {q.options}')
                print(f'  Source: {q.source_quote[:80]}')
                print()
                result['questions'].append(qdata)
            results.append(result)
            print(f'Passed: {elapsed:.1f}s\n')
        except Exception as e:
            elapsed = time.perf_counter() - start
            results.append({
                'count': count, 'difficulty': difficulty,
                'status': 'failed', 'seconds': round(elapsed, 1),
                'error': str(e)
            })
            print(f'Failed in {elapsed:.1f}s: {e}\n')

print('\n=== BENCHMARK SUMMARY ===')
for r in results:
    icon = 'PASS' if r['status'] == 'passed' else 'FAIL'
    print(f"{icon} {r['count']} {r['difficulty']}: {r['seconds']}s")

Path('/content/phi35_benchmark.json').write_text(
    json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8'
)
print('\nResults saved to /content/phi35_benchmark.json')

## Cell 6: Quality Comparison — Check Distractor Ambiguity

The main Qwen2.5-3B problem was ambiguous distractors. Verify Phi-3.5-mini does better.

In [ ]:
import json, time
from ai_quiz_agent.providers import complete_json

ambiguity_test = [
    {'q': 'Which device is an input device?',
     'options': ['keyboard', 'Mouse', 'Monitor', 'Printer']},
    {'q': 'Which process changes liquid water into water vapour?',
     'options': ['Evaporation', 'Freezing', 'Condensation', 'Collection']},
    {'q': 'What does evaporation change liquid water into?',
     'options': ['water vapour', 'steam', 'ice', 'snow']},
]

print('Testing ambiguity detection (the main Qwen failure mode)...')
print('Expected: keyboard AND Mouse are both input devices (indices [0,1])')
print('Expected: water vapour AND steam are the same (indices [0,1])\n')

start = time.perf_counter()
result = complete_json(
    'You check exam questions for ambiguity.',
    'For each question, list ALL option indices (zero-based) that can correctly '
    'answer it. There may be more than one correct answer. Include synonyms and '
    'overlapping answers. Evaluate the wording literally, using general knowledge. '
    'Do not assume the first option is the only answer. '
    'Return {"answers":[[indices for Q1],[indices for Q2],[indices for Q3]]}. '
    + json.dumps(ambiguity_test),
    max_new_tokens=150
)
elapsed = time.perf_counter() - start

print(f'Response ({elapsed:.1f}s):')
print(json.dumps(result, indent=2))

answers = result.get('answers', [])
if len(answers) >= 3:
    q1_correct = 0 in answers[0] and 1 in answers[0]
    q3_correct = 0 in answers[2] and 1 in answers[2]
    if q1_correct and q3_correct:
        print('\nModel correctly identifies ambiguous options!')
        print('This is the key improvement over Qwen2.5-3B.')
    else:
        print('\nModel missed some ambiguities. Review the output above.')
else:
    print('\nUnexpected response format')

## Cell 7: Run Backend Tests

In [ ]:
import subprocess, os
os.chdir('/content/quiz_ai_module')

result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    capture_output=True, text=True, cwd='/content/quiz_ai_module'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    print('\nSome tests failed. Check the output above.')
else:
    print('\nAll tests passed!')

## Cell 8: LoRA Fine-Tuning with Diverse Training Data

**Optional but recommended.** Creates proper training data across multiple subjects and fine-tunes the model.

In [ ]:
import json, os
from pathlib import Path

diverse_train = [
    {"source_id": "science-water-cycle", "passage": "Evaporation changes liquid water into water vapour using heat energy from the Sun. Condensation is the opposite process where water vapour cools and turns into liquid droplets. Precipitation occurs when water droplets in clouds become heavy enough to fall as rain, snow, sleet or hail. Collection happens when precipitation gathers in rivers, lakes and oceans.",
     "question": {"id": "q1", "question": "Which process converts liquid water into water vapour?", "options": ["Evaporation", "Condensation", "Precipitation", "Collection"], "correct_answer": "Evaporation", "explanation": "The passage states that evaporation changes liquid water into water vapour.", "topic": "Water Cycle", "difficulty": "easy", "source_quote": "Evaporation changes liquid water into water vapour using heat energy from the Sun."},
     "provenance": "Authored practice example for science training"},

    {"source_id": "science-water-cycle", "passage": "Evaporation changes liquid water into water vapour using heat energy from the Sun. Condensation is the opposite process where water vapour cools and turns into liquid droplets. Precipitation occurs when water droplets in clouds become heavy enough to fall as rain, snow, sleet or hail. Collection happens when precipitation gathers in rivers, lakes and oceans.",
     "question": {"id": "q1", "question": "What provides the energy that drives evaporation?", "options": ["The Sun", "The Moon", "Wind", "Gravity"], "correct_answer": "The Sun", "explanation": "The passage states heat energy comes from the Sun.", "topic": "Water Cycle", "difficulty": "easy", "source_quote": "Evaporation changes liquid water into water vapour using heat energy from the Sun."},
     "provenance": "Authored practice example for science training"},

    {"source_id": "cs-memory", "passage": "The CPU executes program instructions stored in memory. RAM is volatile memory that stores data for currently running programs. When power is removed, RAM contents are lost. An SSD is non-volatile storage that retains files without power. A hard disk drive uses spinning magnetic platters to store data permanently.",
     "question": {"id": "q1", "question": "Which component executes program instructions?", "options": ["CPU", "RAM", "SSD", "Monitor"], "correct_answer": "CPU", "explanation": "The passage states the CPU executes program instructions.", "topic": "Computer Hardware", "difficulty": "easy", "source_quote": "The CPU executes program instructions stored in memory."},
     "provenance": "Authored practice example for CS training"},

    {"source_id": "cs-memory", "passage": "The CPU executes program instructions stored in memory. RAM is volatile memory that stores data for currently running programs. When power is removed, RAM contents are lost. An SSD is non-volatile storage that retains files without power. A hard disk drive uses spinning magnetic platters to store data permanently.",
     "question": {"id": "q1", "question": "Which type of memory loses its contents when power is removed?", "options": ["RAM", "SSD", "Hard disk drive", "USB flash drive"], "correct_answer": "RAM", "explanation": "The passage says RAM is volatile and its contents are lost when power is removed.", "topic": "Computer Hardware", "difficulty": "easy", "source_quote": "RAM is volatile memory that stores data for currently running programs."},
     "provenance": "Authored practice example for CS training"},

    {"source_id": "pak-geo", "passage": "Pakistan has four provinces: Punjab, Sindh, Khyber Pakhtunkhwa, and Balochistan. Islamabad is the federal capital. Karachi is the largest city by population and serves as the main seaport. The Indus River flows through the length of Pakistan from north to south.",
     "question": {"id": "q1", "question": "Which city is the federal capital of Pakistan?", "options": ["Islamabad", "Karachi", "Lahore", "Peshawar"], "correct_answer": "Islamabad", "explanation": "The passage states Islamabad is the federal capital.", "topic": "Pakistan Geography", "difficulty": "easy", "source_quote": "Islamabad is the federal capital."},
     "provenance": "Authored practice example for geography training"},

    {"source_id": "pak-geo", "passage": "Pakistan has four provinces: Punjab, Sindh, Khyber Pakhtunkhwa, and Balochistan. Islamabad is the federal capital. Karachi is the largest city by population and serves as the main seaport. The Indus River flows through the length of Pakistan from north to south.",
     "question": {"id": "q1", "question": "Which is the largest city in Pakistan by population?", "options": ["Karachi", "Islamabad", "Lahore", "Quetta"], "correct_answer": "Karachi", "explanation": "The passage identifies Karachi as the largest city by population.", "topic": "Pakistan Geography", "difficulty": "easy", "source_quote": "Karachi is the largest city by population and serves as the main seaport."},
     "provenance": "Authored practice example for geography training"},

    {"source_id": "math-percent-1", "passage": "A shop lists a notebook at 1600 rupees. The shop offers a discount of 25 percent of the listed price. The discount amount is 400 rupees. The buyer pays 1200 rupees after the discount.",
     "question": {"id": "q1", "question": "What is the discount amount on the 1600-rupee notebook at 25 percent?", "options": ["400 rupees", "300 rupees", "500 rupees", "200 rupees"], "correct_answer": "400 rupees", "explanation": "1600 multiplied by 25 and divided by 100 equals 400.", "topic": "Percentages", "difficulty": "easy", "source_quote": "The discount amount is 400 rupees."},
     "provenance": "Authored practice example for math training"},

    {"source_id": "math-percent-2", "passage": "A class has 40 learners. Ten learners are absent, which is 25 percent of the class. Thirty learners are present, which is 75 percent of the class.",
     "question": {"id": "q1", "question": "How many learners are present in the class of 40?", "options": ["Thirty", "Twenty", "Forty", "Ten"], "correct_answer": "Thirty", "explanation": "The passage states thirty learners are present.", "topic": "Percentages", "difficulty": "easy", "source_quote": "Thirty learners are present, which is 75 percent of the class."},
     "provenance": "Authored practice example for math training"},

    {"source_id": "islamic-basic", "passage": "Islam has five pillars: Shahada (declaration of faith), Salat (prayer five times daily), Zakat (almsgiving), Sawm (fasting during Ramadan), and Hajj (pilgrimage to Makkah). Muslims pray five times a day: Fajr, Dhuhr, Asr, Maghrib and Isha.",
     "question": {"id": "q1", "question": "How many times a day do Muslims pray?", "options": ["Five", "Three", "Seven", "Four"], "correct_answer": "Five", "explanation": "The passage states Muslims pray five times a day.", "topic": "Islamic Studies", "difficulty": "easy", "source_quote": "Muslims pray five times a day: Fajr, Dhuhr, Asr, Maghrib and Isha."},
     "provenance": "Authored practice example for Islamic Studies training"},

    {"source_id": "gk-solar", "passage": "The solar system has eight planets. Mercury is the closest planet to the Sun. Jupiter is the largest planet. Earth is the only planet known to support life. Mars has two small moons named Phobos and Deimos.",
     "question": {"id": "q1", "question": "Which planet is closest to the Sun?", "options": ["Mercury", "Venus", "Earth", "Mars"], "correct_answer": "Mercury", "explanation": "The passage states Mercury is closest to the Sun.", "topic": "General Knowledge", "difficulty": "easy", "source_quote": "Mercury is the closest planet to the Sun."},
     "provenance": "Authored practice example for GK training"},
]

diverse_validation = [
    {"source_id": "val-bio", "passage": "Photosynthesis is the process by which green plants convert carbon dioxide and water into glucose and oxygen using sunlight. Chlorophyll is the green pigment in leaves that captures light energy. The process takes place mainly in the leaves of plants.",
     "question": {"id": "q1", "question": "Which pigment captures light energy for photosynthesis?", "options": ["Chlorophyll", "Hemoglobin", "Melanin", "Keratin"], "correct_answer": "Chlorophyll", "explanation": "The passage identifies chlorophyll as the green pigment that captures light.", "topic": "Biology", "difficulty": "easy", "source_quote": "Chlorophyll is the green pigment in leaves that captures light energy."},
     "provenance": "Authored practice example for validation"},

    {"source_id": "val-history", "passage": "Pakistan gained independence on 14 August 1947. Muhammad Ali Jinnah served as the first Governor-General. The Lahore Resolution of 1940 demanded a separate Muslim homeland. The first constitution of Pakistan was adopted in 1956.",
     "question": {"id": "q1", "question": "Who was the first Governor-General of Pakistan?", "options": ["Muhammad Ali Jinnah", "Liaquat Ali Khan", "Ayub Khan", "Iskander Mirza"], "correct_answer": "Muhammad Ali Jinnah", "explanation": "The passage states Jinnah served as the first Governor-General.", "topic": "Pakistan History", "difficulty": "easy", "source_quote": "Muhammad Ali Jinnah served as the first Governor-General."},
     "provenance": "Authored practice example for validation"},
]

train_path = Path('/content/quiz_ai_module/training/data/diverse_train.jsonl')
val_path = Path('/content/quiz_ai_module/training/data/diverse_validation.jsonl')

train_path.write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in diverse_train) + '\n', encoding='utf-8')
val_path.write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in diverse_validation) + '\n', encoding='utf-8')

print(f'Created {len(diverse_train)} training examples across {len(set(r["source_id"] for r in diverse_train))} sources')
print(f'Created {len(diverse_validation)} validation examples from separate sources')
print(f'\nSubjects covered: Water Cycle, Computer Hardware, Pakistan Geography, Math, Islamic Studies, General Knowledge')
print(f'Validation subjects: Biology, Pakistan History')

In [ ]:
import json, os, time, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

MODEL_ID = 'microsoft/Phi-3.5-mini-instruct'
OUTPUT_DIR = '/content/quiz-lora-phi35'

train_data_raw = [json.loads(line) for line in
    Path('/content/quiz_ai_module/training/data/diverse_train.jsonl').read_text('utf-8').splitlines() if line.strip()]
val_data_raw = [json.loads(line) for line in
    Path('/content/quiz_ai_module/training/data/diverse_validation.jsonl').read_text('utf-8').splitlines() if line.strip()]

print(f'Training examples: {len(train_data_raw)}')
print(f'Validation examples: {len(val_data_raw)}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = (
    'You create accurate reading-comprehension MCQ quizzes from study material. '
    'Return one JSON object with a "questions" array. Each question needs: id, question, '
    'options (4 distinct choices), correct_answer (must match one option exactly), '
    'explanation, topic, source_quote (exact text from passage), difficulty. '
    'Wrong options must be clearly wrong with no synonyms, umbrella terms or overlapping concepts.'
)

def encode_example(row):
    q = row['question']
    user_msg = f'PASSAGE:\n\"\"\"\n{row["passage"]}\n\"\"\"\n\nGenerate exactly 1 {q["difficulty"]} MCQ. Return only JSON.'
    messages = [
        {'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + user_msg}
    ]
    assistant_content = json.dumps({'questions': [q]}, ensure_ascii=False)
    prompt_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
    all_ids = tokenizer.apply_chat_template(
        messages + [{'role': 'assistant', 'content': assistant_content}],
        tokenize=True
    )
    if len(all_ids) > 2048:
        raise ValueError(f'Example too long: {len(all_ids)} tokens')
    labels = [-100] * len(prompt_ids) + all_ids[len(prompt_ids):]
    return {
        'input_ids': all_ids,
        'attention_mask': [1] * len(all_ids),
        'labels': labels
    }

train_encoded = [encode_example(r) for r in train_data_raw]
val_encoded = [encode_example(r) for r in val_data_raw]
print(f'Encoded: {len(train_encoded)} train, {len(val_encoded)} val')

print(f'\nLoading {MODEL_ID} for LoRA training...')
train_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16,
    trust_remote_code=True, attn_implementation='sdpa'
).to('cuda')
train_model.config.use_cache = False

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['qkv_proj', 'o_proj'],
    task_type='CAUSAL_LM'
)
train_model = get_peft_model(train_model, lora_config)
train_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=20,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=1,
    save_strategy='no',
    eval_strategy='no',
    report_to='none',
    seed=42,
    warmup_steps=2,
)

trainer = Trainer(
    model=train_model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=val_encoded,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)
)

print('\nEvaluating BEFORE training...')
before = trainer.evaluate()
print(f'Before loss: {before["eval_loss"]:.4f}')

print('\nTraining...')
start = time.perf_counter()
train_result = trainer.train()
train_time = time.perf_counter() - start

print(f'\nEvaluating AFTER training...')
after = trainer.evaluate()
print(f'After loss: {after["eval_loss"]:.4f}')

train_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

report = {
    'base_model': MODEL_ID,
    'steps': 20,
    'training_examples': len(train_encoded),
    'validation_examples': len(val_encoded),
    'before_loss': before['eval_loss'],
    'after_loss': after['eval_loss'],
    'improvement': before['eval_loss'] - after['eval_loss'],
    'training_seconds': round(train_time, 1),
    'training_metrics': train_result.metrics,
}
Path(OUTPUT_DIR, 'training_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')

print(f'\n=== TRAINING COMPLETE ===')
print(f'Loss: {before["eval_loss"]:.4f} -> {after["eval_loss"]:.4f} (improvement: {report["improvement"]:.4f})')
print(f'Time: {train_time:.0f}s')
print(f'Adapter saved to: {OUTPUT_DIR}')

del train_model, trainer
torch.cuda.empty_cache()
print('\nTraining model freed from VRAM. Reload inference model in the next cell.')

## Cell 9: Full Interactive Quiz Test

Upload any PDF/DOCX/TXT and test the complete quiz flow.

In [ ]:
import os, sys
from pathlib import Path

project_dir = Path('/content/quiz_ai_module')
os.chdir(project_dir)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

os.environ['AI_PROVIDER'] = 'huggingface'
os.environ['HF_MODEL'] = 'microsoft/Phi-3.5-mini-instruct'

adapter_path = Path('/content/quiz-lora-phi35')
if adapter_path.exists():
    os.environ['HF_ADAPTER'] = str(adapter_path)
    print(f'Using LoRA adapter from {adapter_path}')
else:
    print('Using base Phi-3.5-mini (no LoRA adapter)')

from ai_quiz_agent.providers import get_hf_provider
get_hf_provider.cache_clear()

from ai_quiz_agent.notebook_ui import launch
flow = launch()

## Cell 10: Start FastAPI Server for Flutter App

Starts the backend API with an ngrok tunnel so your Flutter app on a phone can connect.

**Important:** Get a free ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken and paste it below.

In [ ]:
import os, sys, threading
from pathlib import Path

# PASTE YOUR NGROK AUTHTOKEN HERE
NGROK_TOKEN = ''  # Paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_DOMAIN = '' # (Optional) Paste your static domain, e.g. 'my-domain.ngrok-free.app'

project_dir = Path('/content/quiz_ai_module')
os.chdir(project_dir)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

os.environ['AI_PROVIDER'] = 'huggingface'
os.environ['HF_MODEL'] = 'microsoft/Phi-3.5-mini-instruct'
os.environ['DEV_TOKEN'] = os.getenv('DEV_TOKEN', 'colab-dev-token')

adapter_path = Path('/content/quiz-lora-phi35')
if adapter_path.exists():
    os.environ['HF_ADAPTER'] = str(adapter_path)

from ai_quiz_agent.providers import get_hf_provider
get_hf_provider.cache_clear()

def run_server():
    import uvicorn
    from ai_quiz_agent.api.app import app
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(3)

if NGROK_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(8000, domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(8000)
    public_url = tunnel.public_url
    print(f'\n' + '='*60)
    print(f'PUBLIC URL: {public_url}')
    print(f'='*60)
    print(f'\nUse this URL in your Flutter app:')
    print(f'  1. Create .local/flutter-phone.json with:')
    print(f'     {{"API_BASE_URL": "{public_url}"}}')
    print(f'  2. Rebuild the Flutter app with --dart-define-from-file')
    print(f'\nAPI docs: {public_url}/docs')
    print(f'Health check: {public_url}/health')
else:
    print('No ngrok token set. Server running locally on port 8000.')
    print('Set NGROK_TOKEN above to expose the API to your phone.')
    print('\nLocal API docs: http://localhost:8000/docs')

print(f'\nFastAPI server is running. Keep this cell alive.')
print('The server will stop when the Colab runtime disconnects.')

## Cell 11: Download Backup

Download project files and LoRA adapter before the Colab session ends.

In [ ]:
import zipfile, os
from pathlib import Path
from google.colab import files

backup_path = '/content/quiz-checkpoint-phi35.zip'

exclude = {'.git', '__pycache__', '.ipynb_checkpoints', 'node_modules', '.venv'}

with zipfile.ZipFile(backup_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    project = Path('/content/quiz_ai_module')
    for p in sorted(project.rglob('*')):
        if any(part in exclude for part in p.parts):
            continue
        if p.is_file() and p.stat().st_size < 50_000_000:
            zf.write(p, f'quiz_ai_module/{p.relative_to(project)}')

    adapter = Path('/content/quiz-lora-phi35')
    if adapter.exists():
        for p in sorted(adapter.rglob('*')):
            if p.is_file():
                zf.write(p, f'quiz-lora-phi35/{p.relative_to(adapter)}')

    bench = Path('/content/phi35_benchmark.json')
    if bench.exists():
        zf.write(bench, 'phi35_benchmark.json')

print(f'Backup created: {backup_path}')
print(f'Size: {os.path.getsize(backup_path) / 1e6:.1f} MB')

files.download(backup_path)